In [42]:
import os
import sys
import torchvision.transforms as pth_transforms
from argparse import Namespace
from pathlib import Path
import torch
import vision_transformer as vits 
import requests
from PIL import Image
from io import BytesIO
from matplotlib import pyplot as plt


In [43]:

pretrained_weights = Path('/home/mereur1/projects/ocl/ssl_nat_aug/dino/outputs/dino_basic_imgnet/checkpoint.pth')

args = Namespace(
    arch='vit_small',
    patch_size=8,
    pretrained_weights=str(pretrained_weights),
    checkpoint_key='teacher',
    image_path=None,
    image_size=(480, 480),
    output_dir='outputs/visualize_attention/',
    threshold=None
)
print(args)

Namespace(arch='vit_small', patch_size=8, pretrained_weights='/home/mereur1/projects/ocl/ssl_nat_aug/dino/outputs/dino_basic_imgnet/checkpoint.pth', checkpoint_key='teacher', image_path=None, image_size=(480, 480), output_dir='outputs/visualize_attention/', threshold=None)


In [44]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model = vits.__dict__[args.arch](patch_size=args.patch_size, num_classes=0)
for p in model.parameters():
    p.requires_grad = False
model.eval()
model.to(device)

if os.path.isfile(args.pretrained_weights):
    state_dict = torch.load(args.pretrained_weights, map_location="cpu")
    if args.checkpoint_key is not None and args.checkpoint_key in state_dict:
        print(f"Take key {args.checkpoint_key} in provided checkpoint dict")
        state_dict = state_dict[args.checkpoint_key]
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        # remove `backbone.` prefix induced by multicrop wrapper
        state_dict = {k.replace("backbone.", ""): v for k, v in state_dict.items()}
        msg = model.load_state_dict(state_dict, strict=False)
        print('Pretrained weights found at {} and loaded with msg: {}'.format(args.pretrained_weights, msg))

/tmp/ipykernel_3025909/2015642730.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(args.pretrained_weights, map_location="cpu")


Take key teacher in provided checkpoint dict
Pretrained weights found at /home/mereur1/projects/ocl/ssl_nat_aug/dino/outputs/dino_basic_imgnet/checkpoint.pth and loaded with msg: _IncompatibleKeys(missing_keys=['cls_token', 'pos_embed', 'patch_embed.proj.weight', 'patch_embed.proj.bias', 'blocks.0.norm1.weight', 'blocks.0.norm1.bias', 'blocks.0.attn.qkv.weight', 'blocks.0.attn.qkv.bias', 'blocks.0.attn.proj.weight', 'blocks.0.attn.proj.bias', 'blocks.0.norm2.weight', 'blocks.0.norm2.bias', 'blocks.0.mlp.fc1.weight', 'blocks.0.mlp.fc1.bias', 'blocks.0.mlp.fc2.weight', 'blocks.0.mlp.fc2.bias', 'blocks.1.norm1.weight', 'blocks.1.norm1.bias', 'blocks.1.attn.qkv.weight', 'blocks.1.attn.qkv.bias', 'blocks.1.attn.proj.weight', 'blocks.1.attn.proj.bias', 'blocks.1.norm2.weight', 'blocks.1.norm2.bias', 'blocks.1.mlp.fc1.weight', 'blocks.1.mlp.fc1.bias', 'blocks.1.mlp.fc2.weight', 'blocks.1.mlp.fc2.bias', 'blocks.2.norm1.weight', 'blocks.2.norm1.bias', 'blocks.2.attn.qkv.weight', 'blocks.2.attn.

In [45]:
if args.image_path is None:
    # user has not specified any image - we use our own image
    print("Please use the `--image_path` argument to indicate the path of the image you wish to visualize.")
    print("Since no image path have been provided, we take the first image in our paper.")
    response = requests.get("https://dl.fbaipublicfiles.com/dino/img.png")
    img = Image.open(BytesIO(response.content))
    img = img.convert('RGB')
elif os.path.isfile(args.image_path):
    with open(args.image_path, 'rb') as f:
        img = Image.open(f)
        img = img.convert('RGB')
else:
    print(f"Provided image path {args.image_path} is non valid.")
    sys.exit(1)
transform = pth_transforms.Compose([
    pth_transforms.Resize(args.image_size),
    pth_transforms.ToTensor(),
    pth_transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
img = transform(img)

Please use the `--image_path` argument to indicate the path of the image you wish to visualize.
Since no image path have been provided, we take the first image in our paper.


tensor([[[-1.9467, -2.0323, -2.0323,  ..., -2.0152, -2.0152, -2.0152],
         [-1.8953, -1.9980, -2.0152,  ..., -2.0152, -1.9980, -2.0152],
         [-1.8439, -1.9809, -2.0323,  ..., -2.0152, -1.9809, -2.0152],
         ...,
         [-1.4329, -1.3644, -1.3815,  ..., -2.0837, -2.0494, -2.0152],
         [-1.4500, -1.4329, -1.4500,  ..., -2.0837, -2.0323, -2.0152],
         [-1.4843, -1.4672, -1.5014,  ..., -2.0665, -2.0152, -1.9980]],

        [[-1.5980, -1.6856, -1.6856,  ..., -1.6681, -1.6856, -1.6681],
         [-1.5105, -1.6331, -1.6681,  ..., -1.6331, -1.6331, -1.6331],
         [-1.3529, -1.5455, -1.6856,  ..., -1.6155, -1.5980, -1.5980],
         ...,
         [-0.8803, -0.8978, -0.9153,  ..., -1.9307, -1.9132, -1.8957],
         [-0.9153, -0.9153, -0.9503,  ..., -1.9307, -1.8957, -1.8782],
         [-0.9328, -0.9503, -1.0028,  ..., -1.9132, -1.8782, -1.8606]],

        [[-1.4036, -1.4559, -1.4907,  ..., -1.4733, -1.4907, -1.4733],
         [-1.3513, -1.4559, -1.4384,  ..., -1